# 009 - Model tuning

Phase 6 left an open question I never answered: the training run overfit hard past epoch 3, train loss kept dropping while val loss climbed, and I noted at the time that I didn't know whether stronger regularisation could push past that or whether epoch 3 was close to a real ceiling for the setup. Phases 7 and 8 were about measuring what I had. This phase is about trying to beat it.

Before running anything I had to settle how I'd decide whether a change actually helped, because getting that wrong would quietly invalidate everything I'd already reported.

## Selecting on validation, never on test

The test split is what phases 7 and 8 report. If I tune anything against it, those numbers stop being an honest estimate of how the model does on data it has never seen, because I'd have used the test set to make decisions. So every variant here is ranked on the validation split only, and the test split gets touched exactly once at the end, by the single winner.

I also changed what I rank on. Phase 6 selected checkpoints by validation loss, but phase 7 showed that the loss number on its own couldn't tell me whether the model was any good at retrieval, which is the actual task. So the selection metric here is validation Recall@10, averaged over both directions (text to image, image to text) so I'm not accidentally optimising one direction at the other's expense. Implementation is `src/evaluation/validate.py::recall_score`.

Reference point, the phase 6/7 checkpoint scored on validation under this metric:

| metric | value |
|---|---|
| text to image R@10 | 0.4489 |
| image to text R@10 | 0.5498 |
| combined score | 0.4994 |

Its test score was 0.4994 too (0.437 and 0.561), which is a useful sanity check on its own - validation and test agree closely, so validation is a trustworthy thing to select on here.

## Two bugs I found before tuning anything

Neither of these was what I set out to look for, but both had to be fixed before any comparison would mean anything.

The first one matters for reproducibility of numbers I already reported. `RadiologyNETDataset.__getitem__` picked its image with `random.choice(slices)` unconditionally, including for validation and test. That was a deliberate augmentation decision back in phase 4, but I only ever meant it for training. What it actually meant is that every time I recomputed test embeddings, any image with more than one slice got a different picture. That is about 5% of rows, almost all XA. This is a much better explanation for the thing I noticed in phase 7, where re-running the clustering moved image ARI from 0.77 to 0.95, than the bf16 numerical noise I attributed it to at the time. Roughly 140 test images silently changing content between runs will move k-means boundaries a lot more than floating point rounding will. Fixed by adding a `random_slice` flag, defaulting to on for training and forced off everywhere in evaluation. I also sorted `list_slices` output, since `os.listdir` order isn't guaranteed.

That bug had a second effect I only noticed afterwards. `render_examples` in `src/evaluation/inspect.py` has always drawn `slices[0]`, the first slice, while the embeddings being ranked came from a randomly chosen one. So for multi-slice images, phase 7's manual inspection mosaics were potentially showing me a different frame than the one the model actually scored. Everything I concluded from them still stands, since it was all about modality and general anatomy being right, which doesn't change between frames of the same study, but it was a real inconsistency. Forcing evaluation to the first slice makes the two agree by construction.

The second bug is smaller. Weight decay was being applied to every trainable parameter including LayerNorm weights, biases, and `logit_scale`. Decaying those doesn't regularise anything useful, it just pulls the normalisation statistics and the learned temperature toward zero. open_clip's own training script excludes parameters with fewer than 2 dimensions from weight decay, so I did the same in `build_param_groups`. I also gave `logit_scale` its own parameter group pinned to the base learning rate, so that the 10x head multiplier in the variants below can't accidentally be applied to the learned temperature, which is a single scalar and an easy way to destabilise the loss.

## Re-measuring the hardware, because phase 5's numbers had gone stale

I assumed I could just reuse phase 6's batch size of 96. It immediately ran out of memory. The card is 6.44GB but my desktop was holding about 1.2GB of it (Chrome, Discord, Spotify, Steam), leaving under 5GB, and phase 5's measurements were taken when less was open. Worth remembering as a general point: a measured safe batch size isn't a permanent property of the hardware if the GPU is also driving the desktop.

Re-measured sustained (multiple consecutive steps, not one step, which is the mistake phase 6 already caught me making once):

| config | peak VRAM | per step | per sample |
|---|---|---|---|
| batch 48, no checkpointing | 2.68GB | 0.57s | 12.0ms |
| batch 64, no checkpointing | 3.19GB | 0.71s | 11.1ms |
| batch 96, no checkpointing | out of memory | - | - |
| batch 96, gradient checkpointing | 3.46GB | 2.16s | 22.5ms |

Two things came out of this that changed the plan.

Per sample cost is flat at around 11 to 12ms regardless of batch size. So batch size is essentially free in wall clock terms and only limited by memory. That matters more than usual here, because CLIP's contrastive loss learns from in-batch negatives, so a bigger batch is a genuinely harder and more informative training signal rather than just a throughput knob. It made a large-batch variant worth testing.

Gradient checkpointing works but is a poor fit for this specific setup. It cuts memory as advertised, but costs about 1.8x per step, and with partial fine-tuning only the last 2 blocks of each encoder are trainable, so no gradient ever needs to flow back through blocks 0 to 9. Checkpointing recomputes their forward passes for nothing. I only used it for the large-batch variant, where the extra negatives were the point and the memory was otherwise unreachable.

I also found that all of phase 6 had been running with the GPU starved. `num_workers=0` meant image loading and GPU compute were serialised, and loading alone was 0.759s per batch. With `num_workers=4` that drops to 0.148s. Phase 4 had noted that dataloader workers hang on Windows, but that was a Jupyter-specific problem, they work fine from a standalone script as long as the script has an `if __name__ == '__main__'` guard. `num_workers=8` is too many for this machine and dies with a paging file error.

One correction to phase 6's own record while I'm here: its reported 1.2s/step was an undercount. The timing there didn't call `torch.cuda.synchronize()`, and CUDA work is asynchronous, so the timer stopped before the GPU had actually finished. The real figure is the 11 to 12ms per sample above.

## What I actually tried, and why each one

All four variants share the same base: partial fine-tuning, bf16, batch 64, AdamW, cosine schedule with warmup, gradient clipping at 1.0, 8 epochs with patience 3. Eight epochs rather than phase 6's twenty because phase 6 established that the useful improvement here is over by epoch 3 or 4, so a longer budget just burns time. Definitions live in `src/training/experiments.py`.

v1_baseline is the control. Phase 6's settings again, but with the fixes above and the new selection metric, so that the other three are being compared against something measured under identical conditions rather than against a number from a different code path.

v2_head_lr gives the projection heads a learning rate 10x the backbone's. The reasoning is that the two parts have very different jobs. The pretrained transformer blocks already encode useful general biomedical visual and language structure and only need nudging, while the projection heads are the part responsible for mapping into a shared space that fits this particular dataset, and phase 8 gave me direct evidence this matters, since fine-tuning helped the image to text direction noticeably more than text to image, suggesting the text side had further to move. Implemented by splitting the optimiser into parameter groups, which also required fixing the scheduler, since the old one overwrote every group's learning rate with a single value and would have silently flattened the multiplier away.

v3_regularised attacks the overfitting directly, with weight decay raised from 0.1 to 0.5 and only 1 unfrozen block per encoder instead of 2. That roughly halves the trainable transformer capacity. This is the variant most directly aimed at phase 6's open question.

v4_big_batch triples the batch to 192 to get three times the in-batch negatives, which is the lever the CLIP literature leans on hardest. It's the only variant that needs gradient checkpointing to fit, so it's also the slowest, roughly 1.8x per sample.

Worth being explicit that v1 is not identical to phase 6's run even setting the fixes aside. The batch size is 64 rather than 96, and the cosine schedule is stretched over 8 epochs rather than 20, so the learning rate anneals faster. Those are forced by the memory situation and the time budget rather than chosen, but they do mean v1 is the honest control for this phase, not a reproduction of phase 6.

## How big does a difference have to be before I believe it

Easy trap here: run four variants, pick the highest number, declare it an improvement. But Recall@10 is a proportion measured over a finite number of queries, so it comes with sampling error, and with four variants I get four chances to draw a high number by luck.

Validation has 2890 images and 998 unique diagnosis texts, so:

| quantity | n | value | standard error |
|---|---|---|---|
| text to image R@10 | 998 | ~0.45 | 0.0157 |
| image to text R@10 | 2890 | ~0.55 | 0.0093 |
| combined score | - | ~0.50 | 0.0091 |

The combined score's error is the two directions' errors added in quadrature and halved, since it's their mean. Two standard errors is about 0.018, so as a rough bar a variant needs to land above roughly 0.517 before I'd call it a real improvement on the 0.4994 baseline rather than noise. Anything inside that band is a tie as far as I'm concerned, and I'd rather say so than pick a winner by decimal places. Implemented in `src/evaluation/compare_variants.py` so the comparison table prints the error bars next to the numbers instead of leaving me to eyeball it.

This is approximate. It treats queries as independent, which isn't strictly true when several images share one exam, and it doesn't correct for testing four variants at once. Both of those push in the direction of the real bar being wider, not narrower, so if anything I should be more sceptical of small gaps, not less.

## Validation loss and retrieval quality stop agreeing

This showed up in the very first variant and it retroactively justifies changing the selection metric, so it's worth pulling out on its own. v1_baseline, epoch by epoch:

| epoch | train loss | val loss | val R@10 score |
|---|---|---|---|
| 0 | 1.9801 | 2.1770 | 0.4510 |
| 1 | 1.3675 | 2.1240 | 0.4774 |
| 2 | 1.2442 | 2.1227 | 0.4863 |
| 3 | 1.1713 | 2.1222 | 0.4966 |
| 4 | 1.1112 | 2.1298 | 0.5059 |

Validation loss bottoms out at epoch 3 and starts climbing at epoch 4. Retrieval quality does not, it keeps improving. Phase 6 selected checkpoints on validation loss with early stopping, so on this run it would have called epoch 3 the best model and, given enough patience epochs of the same behaviour, eventually stopped. The metric I actually care about says epoch 4 is better than epoch 3.

The reason these can disagree is that they measure different things. The contrastive loss cares about the full similarity distribution, including how confidently wrong pairs are pushed apart, and it's computed within batches of 64. Recall@10 only cares whether a correct match lands somewhere in the top 10 out of the entire split. A model can get more overconfident, which the loss punishes, while its ranking stays the same or improves, which is all Recall@10 sees.

Phase 6's overfitting diagnosis was based on exactly the signal that turns out to be the less relevant one. It wasn't wrong that the loss curve had turned, but "the loss is rising so the model is getting worse at the task" doesn't follow, and here it's directly false for at least one epoch past the turn. Worth saying plainly since phase 6's notebook currently frames the epoch 3 turnaround as the model starting to degrade.

## A crash worth writing down, because the cause wasn't what it looked like

v2 died partway through its first epoch with `OSError [WinError 1455] The paging file is too small`, and at the moment I caught it the disk showed 231MB free out of 476GB. My first read was that the drive was simply full.

That was wrong. Killing the training processes brought it straight back to about 28GB free, and the Windows paging file shrank from 10.13GB to 8.78GB on its own. Nothing had been written to disk and then leaked, the paging file had temporarily expanded to back the memory commitments of the running processes and squeezed everything else out.

The actual cause was my own dataloader configuration. I had given both the training loader and the validation loader `num_workers=4` with `persistent_workers=True`. That is 8 worker processes, each one a separate Windows process carrying its own full torch import at roughly 800MB, all held open for the entire run. The validation loader did not need any of that, it runs about 46 batches once per epoch, and its 4 workers spent the rest of the time idle and expensive.

Fixed by putting validation on `num_workers=0` and dropping the training loader's prefetch from 4 batches to 2. That takes it from 8 worker processes to 4. I also made the experiment runner resume from `results/tuning/summary.json` instead of starting the whole matrix over, which is what it should have done from the beginning, since v1 had completed successfully and its result was nearly thrown away.

Two things to take from this. The first is that on a machine where the GPU and the paging file are both shared with everything else running, resource limits are not fixed properties I can measure once in phase 5 and rely on afterwards. The second is more of a debugging lesson: the error message pointed at the disk, and the disk did genuinely look full, but the disk was a symptom. If I had reacted by deleting files I would have freed space, seen the problem apparently go away, and never found the real cause.

## Batch 192 is not reachable on this GPU, and it fails silently

The large batch variant was meant to run at batch 192, three times v1's negatives. It started, and the first logged step came back at 494 seconds. Expected was about 4.3 seconds. That is roughly 115x slower, which works out to 16.5 hours per epoch, so I killed it.

Peak VRAM was 5.84GB against roughly 5GB actually usable once the desktop's share is accounted for. What happens past that point on Windows is not an out of memory error, it's the WDDM driver quietly backing the excess with system RAM over PCIe. The allocation succeeds, training proceeds, every number in the log looks normal, and the arithmetic just runs at a fraction of the speed. That in turn drove the paging file up until the disk hit 3GB free, which is how I noticed at all, since I had a disk space alarm running by then.

Phase 5 already recorded this exact failure mode, in those words: allocator success does not mean usable, verify with real throughput. I did not apply it here. I picked 192 by interpolating between two measured points instead of measuring it, which is precisely the shortcut that lesson warns against. Worth writing down as a process failure and not just a hardware fact, because the information to avoid it was already in my own notes.

Measured properly the second time, before committing to a run:

| config | peak VRAM | per step | per sample | verdict |
|---|---|---|---|---|
| batch 96, checkpointing | 3.46GB | 2.16s | 22.5ms | fine |
| batch 128, checkpointing | 4.25GB | 2.82s | 22.0ms | fine |
| batch 192, checkpointing | 5.84GB | 494s | 2573ms | spilling to system RAM |

The per sample cost holds steady at about 22ms right up until it doesn't, and then it goes off a cliff rather than degrading gradually. So the large batch variant ran at 128, which is twice the negatives of the batch 64 baseline rather than the three times I planned. That makes it a weaker test of the more-negatives hypothesis than intended, and I'd rather say so than present it as a clean verdict on batch size.

## Results on validation

All four variants finished. Best epoch per variant, ranked on the validation Recall@10 score:

| variant | best epoch | t2i R@10 | i2t R@10 | score | vs baseline |
|---|---|---|---|---|---|
| phase 6/7 checkpoint | - | 0.4489 | 0.5498 | 0.4994 | - |
| v1_baseline | 6 | 0.4619 | 0.5654 | 0.5137 | +0.0143 |
| v2_head_lr | 6 | 0.4589 | 0.5702 | 0.5146 | +0.0152 |
| v3_regularised | 7 | 0.4479 | 0.5498 | 0.4989 | -0.0005 |
| v4_big_batch | 6 | 0.4599 | 0.5588 | 0.5094 | +0.0100 |

Standard error on the score is 0.0091, so the two standard error bar is 0.018. Nothing clears it. Taking each variant honestly:

The 10x head learning rate did nothing. v2 beats v1 by 0.0009, a tenth of the noise bar, and the two curves never separated by more than 0.007 at any epoch. My reasoning was that the projection heads carry the domain shift while the pretrained blocks only need nudging, and phase 8's directional asymmetry seemed to support it. The data does not. That is a negative result and I would rather record it than quietly drop it.

Stronger regularisation actively did not help. v3 lands on the baseline, about 0.015 below v1 and v2. That is the direct answer to the question phase 6 left open, and the answer is no. The reason it fails is the loss/recall divergence above: phase 6 diagnosed overfitting from the validation loss curve, that curve is not tracking retrieval quality, so v3 spent real modelling capacity defending against a problem that was not costing anything on the actual task.

More in-batch negatives helped early and then stopped. v4 was clearly ahead through epoch 3, at 0.5085 while v1 was at 0.4966, and it did that with half the optimiser steps per epoch. Then it flattened and finished slightly behind. That reads as more negatives buying faster convergence per epoch rather than a better final model, but I cannot say it cleanly, because holding epochs fixed while doubling the batch also halves the number of weight updates. Batch size and update count are confounded in this design and separating them would need a run matched on steps rather than epochs.

One more caveat. v1, v2 and v4 all peaked at epoch 6 and declined, but v3 peaked at its very last epoch, so v3 may have been truncated by the 8-epoch budget rather than genuinely topping out, lower capacity models converging more slowly. I would want a longer v3 run before calling it worse rather than just slower.

## Test set results, and a correction to phase 8

v2 was the nominal winner on validation, so that is the one checkpoint that went to test, following the rule I set before running anything rather than reinterpreting it after seeing the numbers.

While doing this I realised the phase 7 and phase 8 test numbers could not be compared against the new ones directly. Those were computed with the random slice bug still in place, so they are not the same measurement. I recomputed both the phase 6/7 checkpoint and zero-shot BiomedCLIP through the identical fixed pipeline. All three rows below are therefore apples to apples, test split, deterministic first slice:

| model | t2i R@10 | i2t R@10 | score | image ARI | text ARI |
|---|---|---|---|---|---|
| zero-shot BiomedCLIP | 0.3333 | 0.3723 | 0.3528 | 0.7188 | 0.6987 |
| phase 6/7 checkpoint | 0.4414 | 0.5611 | 0.5013 | 0.6660 | 0.8597 |
| v2_head_lr, phase 9 | 0.4525 | 0.5726 | 0.5125 | 0.7002 | 0.6587 |

The retrieval story holds and transfers cleanly. v2 gains +0.0112 on test against the phase 6/7 checkpoint, close to the +0.0152 it showed on validation, so the validation gain was real rather than a selection artifact. Fine-tuning as a whole is still doing the heavy lifting: +0.149 over zero-shot, against which phase 9's contribution is small.

The clustering story does not hold, and this is a genuine correction to phase 8. Phase 8 concluded that fine-tuning improves clustering purity on both embedding types. On image embeddings that is false. Zero-shot scores 0.7188 and the fine-tuned phase 6/7 checkpoint scores 0.6660, so fine-tuning slightly degrades how cleanly modality separates in image space. Phase 8 reached the opposite conclusion because it compared a lucky random-slice draw for the fine-tuned model (0.9453) against a random-slice draw for zero-shot (0.7286). Under the fixed pipeline the effect reverses.

That reversal is mechanically sensible in hindsight. Nothing in the training objective asks for modality separation. The contrastive loss optimises alignment between an image and its diagnosis text, and modality structure is incidental structure inherited from BiomedCLIP's pretraining. Pushing embeddings toward text alignment has no reason to preserve it and some reason to blur it.

There is also a real trade-off in v2 that the single selection number hides. Its text embeddings cluster modality much worse than the phase 6/7 checkpoint, 0.6587 against 0.8597, which is below even zero-shot. The 10x head learning rate moved the text projection a long way, and it bought a marginal retrieval gain at a substantial cost to the modality structure of the text space. For this thesis retrieval is the task, so v2 is still defensible as the winner, but calling it simply better than the phase 6/7 checkpoint would be overstating it.

## Two post-hoc attempts: model soup and CSLS

After the four variants finished I tried two things that need no retraining at all, both selected on validation first like everything else.

A uniform model soup, averaging the weights of v1, v2 and v4 (Wortsman et al. 2022, arXiv:2203.05482). These are three fine-tunes of the same pretrained base with different hyperparameters, which is exactly the setting the paper describes, and averaging their weights often beats every individual member. v3 had to be left out, it uses 1 unfrozen block instead of 2 so it does not even have the same set of trainable tensors to average. Implementation in `src/models/soup.py`.

CSLS, a hubness correction applied to the similarity matrix at retrieval time (Conneau et al. 2018, arXiv:1710.04087). In cross modal retrieval a handful of gallery items end up close to many unrelated queries and crowd out the top-k, and CSLS subtracts how generally popular each gallery item is before ranking. Free at inference, no retraining. Implemented as `csls_adjust` in `src/evaluation/retrieval.py`, opt-in via a `csls_k` argument so all the earlier numbers stay reproducible.

CSLS results on validation, sweeping k:

| k | v2_head_lr | soup |
|---|---|---|
| off | 0.5146 | 0.5172 |
| 5 | 0.5143 | 0.5091 |
| 10 | 0.5149 | 0.5180 |
| 20 | 0.5155 | 0.5175 |
| 50 | 0.5113 | 0.5155 |
| 100 | 0.5184 | 0.5165 |

That is noise, not a trend. The values bounce around with no consistent direction, and the k that looks best for v2 (100, +0.0038) is mildly negative for the soup. Picking k=100 on the strength of one column would be exactly the decimal chasing I said I would avoid. CSLS is a negative result here, and I did not carry it forward. My hypothesis was that hubness was costing real recall, and either it is not, or the effect is too small to see against a 0.0091 standard error.

The soup scored 0.5172 on validation, the best number of anything in this phase, beating v2's 0.5146. Small, but it was the validation winner, so under the rule I set at the start it earned the test evaluation.

## The soup won on validation and lost on test, which is the useful part

Everything below is the test split, deterministic slices, one identical pipeline:

| model | val score | test score | t2i R@10 | i2t R@10 | image ARI | text ARI |
|---|---|---|---|---|---|---|
| zero-shot BiomedCLIP | - | 0.3528 | 0.3333 | 0.3723 | 0.7188 | 0.6987 |
| phase 6/7 checkpoint | 0.4994 | 0.5013 | 0.4414 | 0.5611 | 0.6660 | 0.8597 |
| v2_head_lr | 0.5146 | 0.5125 | 0.4525 | 0.5726 | 0.7002 | 0.6587 |
| soup v1+v2+v4 | 0.5172 | 0.5020 | 0.4344 | 0.5696 | 0.6729 | 0.8464 |

The soup was ahead on validation by 0.0026 and behind on test by 0.0105. The ranking flipped completely.

This is the most useful thing to come out of the post-hoc attempts, and it is worth more than either technique. I estimated a two standard error bar of 0.018 on the validation score before running anything, and said differences smaller than that should be treated as ties. The soup is a direct empirical demonstration that this was the right call: a 0.0026 validation lead not only failed to transfer, it reversed into a 0.0105 test deficit. Had I skipped the error analysis and simply crowned whichever checkpoint topped the validation table, I would have shipped the worse model and reported a fictional improvement.

It also retroactively supports how I read the four main variants. v1, v2 and v4 sit within 0.005 of each other on validation, and on this evidence that ordering carries no information at all.

Which model to actually report. v2 was the winner of the pre-registered comparison and gets the test number of 0.5125. The soup was an addition after the fact, it lost on test, and I am not going to now reach back and pick v2 on the basis of its test score, because choosing between them by test performance is precisely the contamination the whole protocol exists to prevent. The defensible statement is that v2 and the soup are indistinguishable, v2 is the model this phase reports because it won the comparison that was declared in advance, and the soup is a documented experiment that did not pan out.

One genuine incidental finding: the soup's text ARI is 0.8464, close to the phase 6/7 checkpoint's 0.8597, while v2 alone collapses to 0.6587. Averaging v2's weights with v1 and v4 undoes most of the damage the 10x head learning rate did to modality structure in the text space. So weight averaging did buy something real, just not on the metric being selected on.

## Two more variants: false negative masking and hard negative batching

These came after the first four, and unlike those they were aimed at things I had actually measured rather than at hyperparameters I guessed might matter.

The first measurement. ClipLoss labels every off-diagonal pair a negative, but images from the same exam carry byte-identical diagnosis text, so some of those negatives are genuinely correct pairs being pushed apart. Counted on the training split: 68,678 identical-text pairs, about 0.52 expected per batch of 64, which means roughly 52 percent of batches contain at least one and about 186 turn up per epoch. The largest single group is 28 images sharing one exact diagnosis. And because identical text produces an identical embedding, the softmax ends up with two indistinguishable logits where only one is labelled correct, which is not something training can fix, it just floors the loss at ln(m) for those rows. `src/training/losses.py::MaskedClipLoss` drops those entries out of the softmax denominator. Verified on a constructed case: the duplicate caps the correct pair's probability at exactly 0.5000, masking restores it to 1.0000, and with no duplicates present the masked loss reduces to plain ClipLoss bit for bit.

The second measurement, and the more interesting one. Using the cached validation embeddings, 98.8 percent of top-10 retrievals already share the query's modality (9858 of 9980). The model has effectively solved modality discrimination. But the modality mix means that in a random batch of 64, only about 14 of the 63 negatives are same-modality, and the other 49 are cross-modality. So roughly 78 percent of the contrastive signal is being spent re-teaching something the model already knows to 98.8 percent. `src/data/sampler.py::ModalityBatchSampler` builds batches from a single modality, which makes all 63 negatives require discriminating anatomy and findings instead. That is about 4.5x more informative negatives at identical memory cost, which is a far bigger lever than the batch size increase v4 paid gradient checkpointing for. Verified: 360 of 361 batches pure at full grouping, every row used exactly once, and the batches reshuffle per epoch.

The two had to ship together. Grouping by modality concentrates same-exam images, and I measured what that does to the false negative rate before running anything: RF-only batches go from 0.52 to 4.29 expected false-negative pairs, XA-only to 3.44. Hard negatives without the mask would have partly cancelled their own benefit.

v5_masked isolates the mask. v6_hardneg is both, with 70 percent of rows in modality-grouped batches and 30 percent left mixed so cross-modality separation does not drift, since the evaluation pool is mixed.

## Six variants, and all of them land in the same place

| variant | best epoch | t2i R@10 | i2t R@10 | score | vs baseline |
|---|---|---|---|---|---|
| phase 6/7 checkpoint | - | 0.4489 | 0.5498 | 0.4994 | - |
| v1_baseline | 6 | 0.4619 | 0.5654 | 0.5137 | +0.0143 |
| v2_head_lr | 6 | 0.4589 | 0.5702 | 0.5146 | +0.0152 |
| v3_regularised | 7 | 0.4479 | 0.5498 | 0.4989 | -0.0005 |
| v4_big_batch | 6 | 0.4599 | 0.5588 | 0.5094 | +0.0100 |
| v5_masked | 7 | 0.4519 | 0.5644 | 0.5081 | +0.0088 |
| v6_hardneg | 6 | 0.4519 | 0.5678 | 0.5099 | +0.0105 |

Still nothing clears the 0.018 bar, and every variant except v3 sits inside a 0.508 to 0.515 band.

Masking on its own did nothing measurable, which is what the arithmetic said it would do. It touches about 1.6 percent of samples per batch, so there was never a large gain available. Its value is correctness, and it was a prerequisite for v6 rather than a candidate in its own right.

Hard negatives are the interesting null. The mechanism plainly engaged. Training loss at epoch 0 was 2.9498 against 1.98 for the variants on random batches, so the task really was much harder by construction, which also means v6's train and validation losses are not comparable with the other rows here. It started behind at epochs 0 and 1, crossed over at epoch 2, and led every other variant through epochs 3 and 4, exactly the shape you would expect if the model were being forced into discriminations that easy batches never demanded. Then it flattened and finished at 0.5099, below v1 and v2.

So the diagnosis was right, the intervention did what it was designed to do, and the result did not move. That is a more informative null than one where the mechanism never engaged at all. If quadrupling the useful negatives changes the learning trajectory visibly but not the endpoint, then the quality of the training signal is not what is binding here.

What that points at is a limit upstream of the optimisation. The translation quality is not the problem, I checked that separately. What is left is the dataset itself. 10,000 exams, about 8,000 of them in training, is small for contrastive learning, and CLIP-style objectives are notoriously data hungry. Six different attempts at extracting more from the same data all converging on the same number is reasonably strong evidence that the data, not the method, is the ceiling.

## What actually came out of this phase

On the question I set out to answer, whether the model can be tuned to do better: marginally, and not by any of the levers I expected.

Eight things were tried. Six training variants (control, layer-wise learning rate, stronger regularisation, larger batch, false negative masking, hard negative batching) and two post-hoc techniques needing no retraining (model soup, CSLS hubness correction). Not one of them beat the phase 6/7 checkpoint by more than the noise bar on validation, and every variant except v3 landed inside a narrow 0.508 to 0.515 band.

The best model this phase produces is v2_head_lr, lifting test Recall@10 from 0.4414 to 0.4525 text to image and 0.5611 to 0.5726 image to text. Real but small, and most of it traces to how checkpoints are selected rather than to any hyperparameter.

Two results are worth more than the tuning outcome itself.

The soup won on validation and lost on test. It led by 0.0026 on validation and trailed by 0.0105 on test, a complete reversal. Before running anything I worked out a two standard error bar of 0.018 and committed to treating anything smaller as a tie. This is direct proof that was the right call, because without it I would have crowned the top validation number, shipped the worse model, and written up an improvement that does not exist.

Hard negatives are an informative null. I measured that 98.8 percent of top-10 retrievals already share the query's modality while only about 14 of 63 negatives in a random batch are same-modality, so roughly 78 percent of the contrastive signal was being spent on a problem already solved. Grouping batches by modality fixed that, and the mechanism visibly engaged, training loss jumped from 1.98 to 2.95 and the variant led every other one through epochs 3 and 4. Then it converged to the same place as everything else. When quadrupling the useful training signal changes the trajectory but not the endpoint, the signal is not what is binding.

Alongside those, the corrections, which are the phase's real output:

The random slice bug meant validation and test were never reproducible, and it invalidated the clustering comparison phase 8's main conclusion rested on. Fixing it reversed that conclusion, zero-shot actually clusters image modality slightly better than the fine-tuned model does.

Validation loss and retrieval quality diverge. Loss bottoms at epoch 3 while Recall@10 keeps climbing to epoch 6, so selecting on loss, which is what phase 6 did, costs about 1.7 points of Recall@10. This is where most of the phase's improvement actually came from.

Weight decay was being applied to LayerNorm weights, biases and logit_scale, which open_clip's own training code deliberately excludes.

The dataloader had been starving the GPU for all of phase 6, image loading taking 0.759s of every 1.2s step.

ClipLoss was treating same-exam images as negatives of each other in about 52 percent of batches, roughly 186 pairs per epoch. Now masked, though it turned out too small to move the metric.

Where this leaves the project. Six independent attempts at extracting more from the same data all converged on the same number, and the translation quality is not the bottleneck. What remains is the dataset. 10,000 exams, about 8,000 of them training, is small for contrastive learning, and CLIP-style objectives are known to be data hungry. I think the honest conclusion is that this checkpoint is close to the ceiling of what this dataset supports, and that further gains would need more data or a different problem formulation rather than more tuning.

Things I did not get to, still worth trying: SigLIP's sigmoid loss, which is reported to work better at small batch sizes and is already available in open_clip. Unfreezing more of the text tower than the vision tower, since phase 8's asymmetry says the text side has further to move and I only tested a higher head learning rate, not more text depth. More trainable capacity rather than less, a direction never explored given v3 showed less is worse. Chunking and averaging the long reports to recover the middle of the 24.9 percent of diagnoses currently truncated away. Image augmentation, the only regularisation lever that adds information instead of removing capacity.

One process note worth keeping. Almost everything of value this phase produced was a bug, a measurement artifact, or a negative result that ruled something out, and two of them had silently corrupted numbers I had already written up as results. Checking that a measurement means what I think it means turned out to be worth considerably more than any hyperparameter I changed.